# 05 Â· Model 4 - DistilBERT fine-tuning
**Owner:** Anvith Choudhary Jasti Â· **Family:** Transformer / self-attention

`distilbert-base-uncased` (Sanh et al., 2019): a 6-layer Transformer encoder distilled from BERT-base, pre-trained on BookCorpus + Wikipedia. The `[CLS]` representation feeds a classification head; **all weights are fine-tuned**.

Why it is in the comparison: self-attention builds **context-dependent** word representations over the whole input window, and large-scale pre-training transfers general language knowledge - the state of the art for this task.

Same protocol as the other models: same cleaned CSV splits, the same 256-token budget (WordPiece tokens incl. `[CLS]`/`[SEP]`), the same metrics. Mixed precision (fp16) keeps it within a 4 GB GPU; on Colab select *Runtime â†’ GPU*.

In [1]:
# ---- Setup: runs locally (from notebooks/) or on Google Colab ----
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Keep data and results on Drive so every notebook in the series sees them.
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/dl-sentiment-project")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
FIG = RESULTS / "figures"
for d in (RAW, PROC, RESULTS, FIG):
    d.mkdir(parents=True, exist_ok=True)
print("project root:", ROOT)

project root: C:\Users\akash\Documents\Study Material\07 Sem7\DL Project


In [2]:
import torch
# ---- Common evaluation protocol (identical in every model notebook) ----
import numpy as np
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)

def compute_metrics(y_true, prob):
    y_true, prob = np.asarray(y_true), np.asarray(prob)
    pred = (prob >= 0.5).astype(int)
    return {"accuracy": accuracy_score(y_true, pred), "precision": precision_score(y_true, pred),
            "recall": recall_score(y_true, pred), "f1": f1_score(y_true, pred),
            "macro_f1": f1_score(y_true, pred, average="macro"), "roc_auc": roc_auc_score(y_true, prob),
            "confusion_matrix": confusion_matrix(y_true, pred).tolist()}

def summarize_runs(runs):
    """Mean and std of test metrics over seeds."""
    keys = ["accuracy", "precision", "recall", "f1", "macro_f1", "roc_auc"]
    vals = {k: [r["test"][k] for r in runs] for k in keys}
    return ({k: float(np.mean(v)) for k, v in vals.items()},
            {k: float(np.std(v)) for k, v in vals.items()})

def save_results(key, result, probs_by_seed, primary_seed=42):
    """results/<key>/metrics.json + per-seed test probabilities (primary seed -> test_probs.npy)."""
    import json
    out = RESULTS / key
    out.mkdir(parents=True, exist_ok=True)
    result["test_mean"], result["test_std"] = summarize_runs(result["runs"])
    (out / "metrics.json").write_text(json.dumps(result, indent=2, default=float))
    for seed, p in probs_by_seed.items():
        np.save(out / f"test_probs_seed{seed}.npy", np.asarray(p, dtype=np.float32))
    np.save(out / "test_probs.npy", np.asarray(probs_by_seed[primary_seed], dtype=np.float32))
    m, s = result["test_mean"], result["test_std"]
    print(f"[{key}] test over {len(result['runs'])} run(s): acc {m['accuracy']:.4f} +/- {s['accuracy']:.4f} | "
          f"F1 {m['f1']:.4f} +/- {s['f1']:.4f} | AUC {m['roc_auc']:.4f}")

In [3]:
import copy, json, random, time
import pandas as pd
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

CHECKPOINT = "distilbert-base-uncased"
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "")
tok = AutoTokenizer.from_pretrained(CHECKPOINT)
splits = {s: pd.read_csv(PROC / f"{s}.csv") for s in ("train", "val", "test")}
y = {s: df["label"].values.astype(np.float32) for s, df in splits.items()}

# Tokenise once without truncation, then build fixed-length inputs for each truncation strategy.
t0 = time.time()
ids = {s: tok(df["text"].tolist(), add_special_tokens=False, truncation=False)["input_ids"] for s, df in splits.items()}
print(f"tokenised in {time.time() - t0:.0f}s; median WordPiece length (train): {int(np.median([len(i) for i in ids['train']]))}")

def build_inputs(seqs, max_len, strategy):
    body = max_len - 2
    x = np.full((len(seqs), max_len), tok.pad_token_id, dtype=np.int64)
    mask = np.zeros((len(seqs), max_len), dtype=np.int64)
    for i, s in enumerate(seqs):
        if len(s) > body:
            s = s[: body // 2] + s[-(body - body // 2):] if strategy == "head_tail" else s[:body]
        s = [tok.cls_token_id] + s + [tok.sep_token_id]
        x[i, : len(s)] = s; mask[i, : len(s)] = 1
    return x, mask

_cache = {}
def get_data(max_len, strategy):
    if (max_len, strategy) not in _cache:
        _cache[(max_len, strategy)] = {s: (*build_inputs(ids[s], max_len, strategy), y[s]) for s in splits}
    return _cache[(max_len, strategy)]

C:\Users\akash\Documents\Study Material\07 Sem7\DL Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda NVIDIA GeForce RTX 3050 Laptop GPU


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (550 > 512). Running this sequence through the model will result in indexing errors


tokenised in 15s; median WordPiece length (train): 213


## Fine-tuning loop
AdamW with weight decay 0.01, linear warm-up over the first 10 % of steps then linear decay, fp16 autocast with gradient scaling, gradient clipping at 1.0, BCE on a single logit. Validation F1 after every epoch; the best epoch is restored.

In [4]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def predict(model, dl):
    model.eval(); out = []
    for xb, mb, _ in dl:
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV.type == "cuda"):
            logits = model(input_ids=xb.to(DEV), attention_mask=mb.to(DEV)).logits.squeeze(-1)
        out.append(torch.sigmoid(logits.float()).cpu())
    return torch.cat(out).numpy()

def run(cfg, seed=42):
    set_seed(seed)
    data = get_data(cfg["max_len"], cfg["truncation"])
    mk = lambda s, shuffle, bs: DataLoader(TensorDataset(*(torch.from_numpy(a) for a in data[s])), batch_size=bs,
                                           shuffle=shuffle, generator=torch.Generator().manual_seed(seed))
    dl = {"train": mk("train", True, cfg["batch_size"]), "val": mk("val", False, 64), "test": mk("test", False, 64)}
    model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=1).to(DEV)
    no_decay = ("bias", "LayerNorm.weight")
    groups = [{"params": [p for n, p in model.named_parameters() if not n.endswith(no_decay)], "weight_decay": 0.01},
              {"params": [p for n, p in model.named_parameters() if n.endswith(no_decay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups, lr=cfg["lr"])
    steps = len(dl["train"]) * cfg["epochs"]
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler(enabled=DEV.type == "cuda")
    loss_fn = nn.BCEWithLogitsLoss()
    best, best_state, history = -1, None, []
    t0 = time.time()
    for epoch in range(1, cfg["epochs"] + 1):
        model.train(); tot = 0.0
        for xb, mb, yb in dl["train"]:
            xb, mb, yb = xb.to(DEV), mb.to(DEV), yb.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV.type == "cuda"):
                logits = model(input_ids=xb, attention_mask=mb).logits.squeeze(-1)
            loss = loss_fn(logits.float(), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(yb)
        p_val = predict(model, dl["val"])
        val_loss = float(nn.functional.binary_cross_entropy(torch.tensor(p_val).clamp(1e-6, 1 - 1e-6), torch.tensor(y["val"])))
        m = compute_metrics(y["val"], p_val)
        history.append({"epoch": epoch, "train_loss": tot / len(y["train"]), "val_loss": val_loss,
                        "val_acc": m["accuracy"], "val_f1": m["f1"], "elapsed_s": time.time() - t0})
        print(f"  epoch {epoch} train_loss {history[-1]['train_loss']:.4f} val_loss {val_loss:.4f} "
              f"val_acc {m['accuracy']:.4f} val_f1 {m['f1']:.4f} ({time.time() - t0:.0f}s)")
        if m["f1"] > best:
            best, best_state = m["f1"], {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    train_time = time.time() - t0
    model.load_state_dict(best_state)
    val = compute_metrics(y["val"], predict(model, dl["val"]))
    t1 = time.time(); p_test = predict(model, dl["test"]); infer_time = time.time() - t1
    n_params = sum(p.numel() for p in model.parameters())
    del model, opt; torch.cuda.empty_cache()
    return {"seed": seed, "config": cfg, "val": val, "test": compute_metrics(y["test"], p_test), "history": history,
            "best_epoch": max(history, key=lambda h: h["val_f1"])["epoch"], "train_time_s": train_time,
            "infer_time_s": infer_time, "total_params": n_params, "trainable_params": n_params}, p_test

def row(name, r):
    return {"variant": name, "val_f1": r["val"]["f1"], "val_acc": r["val"]["accuracy"], "test_acc": r["test"]["accuracy"],
            "test_f1": r["test"]["f1"], "best_epoch": r["best_epoch"], "train_time_s": round(r["train_time_s"], 1)}

## Experiments
Fine-tuning is ~2 orders of magnitude more expensive than the GloVe models, so the search is smaller:
* **Stage 1:** learning rate âˆˆ {2e-5, 5e-5} (head truncation, 256 tokens, 2 epochs, batch 16);
* **Stage 2:** head+tail truncation with the selected learning rate;
* **Stage 3:** the best variant (val F1) re-trained with seeds 42, 43, 44.

A 512-token run does not fit the 4 GB GPU at batch 16 and is outside the common 256-token protocol, so it is not run.

In [5]:
SEEDS = [42, 43, 44]
BASE = {"max_len": 256, "truncation": "head", "lr": 2e-5, "epochs": 2, "batch_size": 16}
runs_cache, search_rows, ablation_rows = {}, [], []

def run_cached(name, cfg, seed=42):
    key = (json.dumps(cfg, sort_keys=True), seed)
    if key not in runs_cache:
        print(f"> {name} (seed {seed})")
        runs_cache[key] = run(cfg, seed)
    return runs_cache[key]

SEARCH = {"lr 2e-5": BASE, "lr 5e-5": {**BASE, "lr": 5e-5}}
for name, cfg in SEARCH.items():
    search_rows.append(row(name, run_cached(name, cfg)[0]))
search_df = pd.DataFrame(search_rows).sort_values("val_f1", ascending=False)
display(search_df)
best_name = search_df.iloc[0]["variant"]; best_cfg = SEARCH[best_name]

> lr 2e-5 (seed 42)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:  18%|â–ˆâ–Š        | 18/100 [00:00<00:00, 156.91it/s]

Loading weights: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 100/100 [00:00<00:00, 601.53it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Loading weights: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 100/100 [00:00<00:00, 586.12it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1 train_loss 0.3242 val_loss 0.2614 val_acc 0.9052 val_f1 0.9035 (256s)


  epoch 2 train_loss 0.1558 val_loss 0.3033 val_acc 0.9116 val_f1 0.9106 (502s)


,variant,val_f1,val_acc,test_acc,test_f1,best_epoch,train_time_s
0,lr 2e-5,0.917372,0.9180,0.91648,0.916077,2,518.6
1,lr 5e-5,0.910563,0.9116,0.91244,0.911638,2,502.1


In [6]:
variants = {f"head 256, {best_name} (selected)": best_cfg,
            f"head+tail 256, {best_name}": {**best_cfg, "truncation": "head_tail"}}
for name, cfg in variants.items():
    ablation_rows.append(row(name, run_cached(name, cfg)[0]))
ablation_df = pd.DataFrame(ablation_rows)
display(ablation_df)
final_name = ablation_df.sort_values("val_f1", ascending=False).iloc[0]["variant"]
final_cfg = variants[final_name]
print("final configuration:", final_name, final_cfg)

> head+tail 256, lr 2e-5 (seed 42)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:  13%|â–ˆâ–Ž        | 13/100 [00:00<00:01, 62.79it/s]

Loading weights: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 100/100 [00:00<00:00, 468.97it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1 train_loss 0.2998 val_loss 0.2507 val_acc 0.8988 val_f1 0.9023 (241s)


  epoch 2 train_loss 0.1556 val_loss 0.2725 val_acc 0.9196 val_f1 0.9190 (481s)


,variant,val_f1,val_acc,test_acc,test_f1,best_epoch,train_time_s
0,"head 256, lr 2e-5 (selected)",0.917372,0.9180,0.91648,0.916077,2,518.6
1,"head+tail 256, lr 2e-5",0.918984,0.9196,0.92872,0.928646,2,481.6


final configuration: head+tail 256, lr 2e-5 {'max_len': 256, 'truncation': 'head_tail', 'lr': 2e-05, 'epochs': 2, 'batch_size': 16}


In [7]:
runs, probs = [], {}
for seed in SEEDS:
    r, p = run_cached(final_name, final_cfg, seed)
    runs.append(r); probs[seed] = p
    print(f"seed {seed}: test acc {r['test']['accuracy']:.4f} F1 {r['test']['f1']:.4f}")

save_results("distilbert", {
    "model": "DistilBERT (fine-tuned)", "owner": "Anvith Choudhary Jasti", "family": "Transformer / attention",
    "final_variant": final_name, "config": {**final_cfg, "checkpoint": CHECKPOINT, "optimizer": "AdamW",
                                            "weight_decay": 0.01, "warmup": 0.1, "fp16": True},
    "search": search_rows, "ablations": ablation_rows,
    "total_params": runs[0]["total_params"], "trainable_params": runs[0]["trainable_params"], "runs": runs,
}, probs)

seed 42: test acc 0.9287 F1 0.9286
> head+tail 256, lr 2e-5 (seed 43)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:  16%|â–ˆâ–Œ        | 16/100 [00:00<00:00, 146.36it/s]

Loading weights: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 100/100 [00:00<00:00, 531.37it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Loading weights: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 100/100 [00:00<00:00, 563.77it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1 train_loss 0.2995 val_loss 0.2604 val_acc 0.9052 val_f1 0.9010 (241s)


  epoch 2 train_loss 0.1546 val_loss 0.2686 val_acc 0.9184 val_f1 0.9199 (482s)


seed 44: test acc 0.9282 F1 0.9296
[distilbert] test over 3 run(s): acc 0.9288 +/- 0.0005 | F1 0.9292 +/- 0.0004 | AUC 0.9800


## Truncation policy

Long reviews are truncated consistently during training and evaluation. Record the selected maximum sequence length with the run configuration so comparisons remain reproducible across seeds.


## Truncation policy

Record the selected maximum sequence length with every run so comparisons remain reproducible.
